# AI 102 - Midterm Project
## Sentiment Classification on Amazon Fine Food Reviews

**Course:** AI 102 - Natural Language-Based Programming Techniques

**Name:**  Reezy Hudson

**Date:**  March 17

**Checkpoint (Phase 1):** March 18

**Final Submission:** Sunday, March 29

---

## Setup

- Add any libraries you might need.
- Make sure to set the path to your midterm project in Google Drive where you stored the data.



In [ ]:
# ── Mount Drive and set paths ──
from google.colab import drive
drive.mount('/content/drive')

import os
# UPDATE THIS PATH if your folder is different
MIDTERM_DIR = '/content/drive/MyDrive/AI102 Midterm'
DATA_DIR = MIDTERM_DIR # The CSV is directly in this directory

# ── Imports ──
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)

print('All imports loaded.')

Mounted at /content/drive
All imports loaded.


---

# Phase 1: Data Exploration & Preparation

---

## Step 1: Load & Explore

Load the dataset and answer these four questions:
1. How many reviews are in the dataset?
2. What is the star rating distribution (1–5)?
3. What is the average review length?
4. Are there any missing values?

**Expected output:**
```
Reviews: ~45,164  |  Columns: 10
5-star: ~64%  |  1-star: ~9%
Avg length: ~437 chars / ~80 words
Missing: Text = 0
```

In [ ]:
# TODO: Load the CSV file
df = pd.read_csv(f'{DATA_DIR}/Reviews_downsampled_25MB.csv')

# TODO: Print total number of reviews and columns
print('Total reviews:', df.shape[0])
print('Total columns:', df.shape[1])

# TODO: Show star rating distribution with counts and percentages
score_counts = df['Score'].value_counts().sort_index()
score_percents = df['Score'].value_counts(normalize=True).sort_index()

print('\nScore Distribution (Counts):')
print(score_counts)
print('\nScore Distribution (Percents):')
print(score_percents.round(2))

# TODO: Calculate and print average review length
df['Review_length'] = df['Text'].astype(str).apply(len)

print('\nAverage Review Length:')
print(round(df['Review_length'].mean(), 2))

# TODO: Check for missing values
print('\nMissing Values:')
print(df.isnull().sum())

Total reviews: 45164
Total columns: 10

Score Distribution (Counts):
Score
1     4196
2     2331
3     3256
4     6424
5    28957
Name: count, dtype: int64

Score Distribution (Percents):
Score
1    0.09
2    0.05
3    0.07
4    0.14
5    0.64
Name: proportion, dtype: float64

Average Review Length:
437.36

Missing Values:
Id                        0
ProductId                 0
UserId                    0
ProfileName               4
HelpfulnessNumerator      0
HelpfulnessDenominator    0
Score                     0
Time                      0
Summary                   3
Text                      0
Review_length             0
dtype: int64


There are 45164 reviews.
The star rating distribution is 1 star-9%, 2 star-5%, 3 star-7%, 4 star-14%, 5 star-64%.
The average review length is 437.36 characters.
There are 7 missing values.

## Step 2: Create Binary Sentiment Labels

- Drop all 3-star reviews (ambiguous)
- Map 4–5 stars to 1 (positive), 1–2 stars to 0 (negative)
- Print counts to verify

**Expected output:**
```
After dropping 3s: ~41,908 reviews
Positive: ~35,381 (84.4%)
Negative: ~6,527 (15.6%)
```

In [ ]:
# TODO: Filter out 3-star reviews (remember .copy())
df_sent = df[df['Score'] != 3].copy()

# TODO: Create binary sentiment labels
binary_labels = df_sent['Score'].apply(lambda x: 1 if x >= 4 else 0)

# TODO: Create a 'sentiment' column (1 = positive, 0 = negative)
df_sent['sentiment'] = binary_labels

# TODO: Print counts to verify
print("Sentiment Counts:")
print(df_sent['sentiment'].value_counts())

print("\nVerification Table:")
print(pd.crosstab(df_sent['Score'], df_sent['sentiment']))


Sentiment Counts:
sentiment
1    35381
0     6527
Name: count, dtype: int64

Verification Table:
sentiment     0      1
Score                 
1          4196      0
2          2331      0
4             0   6424
5             0  28957


## Step 3: Clean the Text

Write a cleaning function that does all of the following:
1. Remove HTML tags (regex recommended but any method works)
2. Convert to lowercase
3. Remove punctuation and numbers
4. Remove stopwords
5. Apply stemming OR lemmatization (choose one — lemmatization recommended)

Then apply it to the entire Text column and show 3 before/after examples.

**Expected:** Zero empty reviews after cleaning. All HTML, uppercase, punctuation removed. Show a few before and after samples.

**Markdown:** Explain which normalization method (stemming or lemmatization) you chose and why.

In [ ]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# TODO: Write your cleaning function
def clean_text(text):
    text = str(text)

    text = re.sub(r"<.*?>", " ", text)

    text = text.lower()

    text = text.translate(str.maketrans("", "", string.punctuation))

    text = re.sub(r"\d+", "", text)

    words = text.split()

    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]

    return " ".join(words)

# TODO: Apply to the cleaned text column
df_sent['clean_text'] = df_sent['Text'].apply(clean_text)

# TODO: Print 3 before/after examples
for i in range(3):
    print("\nORIGINAL:", df_sent['Text'].iloc[i][:120])
    print("CLEANED :", df_sent['clean_text'].iloc[i][:120])

# TODO: Check for empty reviews after cleaning
empty_reviews = (df_sent['clean_text'] == "").sum()
print("\nEmpty reviews after cleaning:", empty_reviews)


ORIGINAL: I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The produc
CLEANED : bought several vitality canned dog food product found good quality product look like stew processed meat smell better la

ORIGINAL: This taffy is so good.  It is very soft and chewy.  The flavors are amazing.  I would definitely recommend you buying it
CLEANED : taffy good soft chewy flavor amazing would definitely recommend buying satisfying

ORIGINAL: I don't know if it's the cactus or the tequila or just the unique combination of ingredients, but the flavour of this ho
CLEANED : dont know cactus tequila unique combination ingredient flavour hot sauce make one kind picked bottle trip brought back h

Empty reviews after cleaning: 0


*Explain your normalization choice (stemming or lemmatization) below:*

I chose lemmatization because it preserves the meaning of the words in context better than stemming does. The goal of the analysis in this notebook includes sentiment analysis, which means the words need to maintain their context in order to be able to dissect their sentiment. Lemmatization preserves information needed to interperate meaning, while stemming does not.

## Step 4A: Sparse Vectorization (BoW + TF-IDF)

Create two feature matrices using your cleaned text.

**Expected output:**
```
BoW shape:    (41908, N)    — N depends on your max_features choice
TF-IDF shape: (41908, N)
```

Each row is an entire review (document-level vectorization).

**Markdown:** Explain what the rows and columns represent.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# TODO: Create BoW matrix with CountVectorizer
bow_vectorizer = CountVectorizer(
    stop_words='english',
    max_features=5000
)

X_bow = bow_vectorizer.fit_transform(df_sent['clean_text'])

# TODO: Create TF-IDF matrix with TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X_tfidf = tfidf_vectorizer.fit_transform(df_sent['clean_text'])

# TODO: Print shapes of both matrices
print("BoW shape:", X_bow.shape)
print("TF-IDF shape:", X_tfidf.shape)


BoW shape: (41908, 5000)
TF-IDF shape: (41908, 5000)


*Explain what the rows and columns represent:*

Each row represents a review. There are 41,908 reviews after filtering.

Each column represents a unique word found. There are 5000 unique words found after vectorization.

## Step 4B: Dense Embeddings — Choose 1 of 4

Choose **one** embedding method. Train your own or use a pre-trained model.

| Method | Train or Pre-trained | Doc Vector Approach |
|--------|---------------------|--------------------|
| Word2Vec (recommended) | Either | Average word vectors per review |
| FastText | Either | Average word vectors per review |
| Doc2Vec | Train only | Direct document vectors |
| BERT | Pre-trained only | [CLS] token or mean pooling |

See the pre-trained models reference table on Canvas for download options.

**Expected output:**
```
Document vector matrix shape: (41908, D)    — D depends on your vector_size
```

**Markdown:** Explain (1) which method you chose and why, (2) whether you trained or used pre-trained and why, (3) how you created document vectors.

In [ ]:
!pip install gensim
from gensim.models import Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 66.1 MB/s eta 0:00:00


In [ ]:
# TODO: Choose and implement your embedding method
tokenized_reviews = df_sent['clean_text'].apply(lambda x: x.split())

w2v_model = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

# TODO: Create document vectors for all reviews
def document_vector(tokens, model):
    word_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(word_vectors, axis=0)

X_embed = np.array([document_vector(tokens, w2v_model) for tokens in tokenized_reviews])

# TODO: Print the shape of your document vector matrix
print("Document vector matrix shape:", X_embed.shape)

Document vector matrix shape: (41908, 100)


*Explain: (1) which method and why, (2) trained vs pre-trained, (3) how you created document vectors:*

Word2Vec was chosen to capture semantics between words.

A trained Word2Vec model was chosen because the dataset has vocabulary that is specifically related to the recorded data surrounding food products and reviews on customer experiences. Training allows for the creation of relationships and patterns specific to this dataset for higher effectiveness.

In order to create document level vectors, each review was tokenized to create individual words, then the tokens had a word embedding identified from the trained Word2Vec. The computed document vector was found through the average of all vectors in the review.

### Phase 1 — Check Your Work

Before moving to Phase 2, verify you have:

| Item | Expected |
|------|----------|
| `df_binary` rows | ~41,908 |
| Positive / Negative | ~35,381 / ~6,527 |
| Empty reviews after cleaning | 0 |
| BoW matrix shape | (41908, N) |
| TF-IDF matrix shape | (41908, N) |
| Embedding matrix shape | (41908, D) |

If your numbers are close to these, you are ready for Phase 2.

---

# Phase 2: Classification & Evaluation

---

## Step 5: Train/Test Split

Split all three feature matrices (BoW, TF-IDF, your embedding) using the **same random_state**.

**Expected output:**
```
Train: ~33,526  |  Test: ~8,382
```

**Markdown:** Explain why we split the data into training and testing sets.

In [ ]:
from sklearn.model_selection import train_test_split

# Define the target variable y
y = df_sent['sentiment']

# TODO: Split all feature matrices with the same random_state
RANDOM_STATE = 42

Xbow_train, Xbow_test, y_train, y_test = train_test_split(
    X_bow, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

Xtfidf_train, Xtfidf_test, _, _ = train_test_split(
    X_tfidf, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

Xemb_train, Xemb_test, _, _ = train_test_split(
    X_embed, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)


# TODO: Print training and testing set sizes
print("BoW train size:", Xbow_train.shape)
print("BoW test size:", Xbow_test.shape)

print("TFIDF train size:", Xtfidf_train.shape)
print("TFIDF test size:", Xtfidf_test.shape)

print("Embedding train size:", Xemb_train.shape)
print("Embedding test size:", Xemb_test.shape)

print("y train size:", y_train.shape)
print("y test size:", y_test.shape)


BoW train size: (33526, 5000)
BoW test size: (8382, 5000)
TFIDF train size: (33526, 5000)
TFIDF test size: (8382, 5000)
Embedding train size: (33526, 100)
Embedding test size: (8382, 100)
y train size: (33526,)
y test size: (8382,)


*Explain why we split the data:*

Split training and testing sets shows how well my embedding worked. The test set being its own entity shows the performance in a way that is uneffected and therefore unbiased, accurately evaluting effectiveness.

## Step 6: Train Classifiers (Binary)

Train these three models:
1. Naive Bayes on BoW
2. Naive Bayes on TF-IDF
3. Logistic Regression on your embedding

You may train additional classifiers — explain why you chose them.

**Markdown:** Explain why Naive Bayes cannot be used on embedding features.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

# TODO: Train Naive Bayes on BoW
nb_bow = MultinomialNB()
nb_bow.fit(Xbow_train, y_train)

# TODO: Train Naive Bayes on TF-IDF
nb_tfidf = MultinomialNB()
nb_tfidf.fit(Xtfidf_train, y_train)

# TODO: Train Logistic Regression on your embedding
lr_embed = LogisticRegression(max_iter=1000, random_state=42)
lr_embed.fit(Xemb_train, y_train)


LogisticRegression(max_iter=1000, random_state=42)

*Explain why Naive Bayes cannot be used on embedding features:*

Naive Bayes says that features are conditionally indpendent, but embedding vectors has interacting dimenions. Naive Bayes would cause improperly relationship assumptions and estimates if taken in the context of independence.

## Step 7: Evaluate & Compare

For each model, show:
- Confusion matrix (heatmap or printed)
- Classification report (precision, recall, F1 per class)
- Accuracy AND balanced accuracy

Compare sparse vs. dense feature performance.

**Remember:** A model that always predicts positive gets ~84% accuracy but only 50% balanced accuracy. Report both.

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score
)

y_pred_nb_bow   = nb_bow.predict(Xbow_test)
y_pred_nb_tfidf = nb_tfidf.predict(Xtfidf_test)
y_pred_lr_embed = lr_embed.predict(Xemb_test)

# TODO: Evaluate each model — confusion matrix + classification report
print("Naive Bayes — BoW Confusion Matrix")
print(confusion_matrix(y_test, y_pred_nb_bow))

print("\nNaive Bayes — TFIDF Confusion Matrix")
print(confusion_matrix(y_test, y_pred_nb_tfidf))

print("\nLogistic Regression — Embedding Confusion Matrix")
print(confusion_matrix(y_test, y_pred_lr_embed))

print("\nNaive Bayes — BoW Report")
print(classification_report(y_test, y_pred_nb_bow))

print("\nNaive Bayes — TFIDF Report")
print(classification_report(y_test, y_pred_nb_tfidf))

print("\nLogistic Regression — Embedding Report")
print(classification_report(y_test, y_pred_lr_embed))

# TODO: Report accuracy and balanced accuracy for each
acc_nb_bow   = accuracy_score(y_test, y_pred_nb_bow)
bal_nb_bow   = balanced_accuracy_score(y_test, y_pred_nb_bow)

acc_nb_tfidf = accuracy_score(y_test, y_pred_nb_tfidf)
bal_nb_tfidf = balanced_accuracy_score(y_test, y_pred_nb_tfidf)

acc_lr_embed = accuracy_score(y_test, y_pred_lr_embed)
bal_lr_embed = balanced_accuracy_score(y_test, y_pred_lr_embed)

# TODO: Create a summary comparison table
summary = pd.DataFrame({
    "Model": [
        "Naive Bayes (BoW)",
        "Naive Bayes (TF-IDF)",
        "Logistic Regression (Embedding)"
    ],
    "Accuracy": [
        acc_nb_bow,
        acc_nb_tfidf,
        acc_lr_embed
    ],
    "Balanced Accuracy": [
        bal_nb_bow,
        bal_nb_tfidf,
        bal_lr_embed
    ]
})

print("\n===== Model Comparison =====")
print(summary)

Naive Bayes — BoW Confusion Matrix
[[ 860  429]
 [ 387 6706]]

Naive Bayes — TFIDF Confusion Matrix
[[ 257 1032]
 [  22 7071]]

Logistic Regression — Embedding Confusion Matrix
[[ 636  653]
 [ 228 6865]]

Naive Bayes — BoW Report
              precision    recall  f1-score   support

           0       0.69      0.67      0.68      1289
           1       0.94      0.95      0.94      7093

    accuracy                           0.90      8382
   macro avg       0.81      0.81      0.81      8382
weighted avg       0.90      0.90      0.90      8382


Naive Bayes — TFIDF Report
              precision    recall  f1-score   support

           0       0.92      0.20      0.33      1289
           1       0.87      1.00      0.93      7093

    accuracy                           0.87      8382
   macro avg       0.90      0.60      0.63      8382
weighted avg       0.88      0.87      0.84      8382


Logistic Regression — Embedding Report
              precision    recall  f1-score   su

## Step 7B: Go Deeper — Choose One

Complete **one** of the following options. Delete the one you do not choose.

---

### Option A: Five-Class Classification
- Pick one feature type (sparse or dense)
- Train any classifier on the full dataset (all 5 star ratings — you will need to re-vectorize)
- Show confusion matrix + classification report
- Compare to your binary results: what got harder?

### Option B: Embedding Visualization
- Visualize your word embeddings using TensorBoard or t-SNE/PCA
- Label or highlight interesting clusters
- Describe what patterns you observe

In [ ]:
# Option A: Five-Class Classification
from sklearn.feature_extraction.text import TfidfVectorizer
# TODO: Re-vectorize using the full dataset (all 5 star ratings — you will need to re-vectorize)
y5 = df["Score"]   # target = 1–5 star ratings

# Ensure 'clean_text' column exists in df for full dataset classification
# The clean_text function is defined in cell HqQIlUJ7D1qN
df['clean_text'] = df['Text'].apply(clean_text)

tfidf5 = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2
)

X5 = tfidf5.fit_transform(df["clean_text"])
print("Matrix shape:", X5.shape)

from sklearn.model_selection import train_test_split

X5_train, X5_test, y5_train, y5_test = train_test_split(
    X5,
    y5,
    test_size=0.2,
    random_state=42,
    stratify=y5
)

# TODO: Train a classifier and evaluate
from sklearn.linear_model import LogisticRegression

model5 = LogisticRegression(max_iter=2000, random_state=42)
model5.fit(X5_train, y5_train)

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y5_pred = model5.predict(X5_test)

print("Confusion Matrix:")
print(confusion_matrix(y5_test, y5_pred))

print("\nClassification Report:")
print(classification_report(y5_test, y5_pred))

acc5 = accuracy_score(y5_test, y5_pred)
print("\nFive-Class Accuracy:", acc5)


Matrix shape: (45164, 20000)
Confusion Matrix:
[[ 470   27   27   27  288]
 [ 116   47   48   38  217]
 [  58   19  116  111  347]
 [  24    3   51  261  946]
 [  41    3   22  130 5596]]

Classification Report:
              precision    recall  f1-score   support

           1       0.66      0.56      0.61       839
           2       0.47      0.10      0.17       466
           3       0.44      0.18      0.25       651
           4       0.46      0.20      0.28      1285
           5       0.76      0.97      0.85      5792

    accuracy                           0.72      9033
   macro avg       0.56      0.40      0.43      9033
weighted avg       0.67      0.72      0.67      9033


Five-Class Accuracy: 0.7184766965570685


Compare What Got Harder

The 5-Class Accuracy had to distinguish small differences in sentiment (not bad vs okay) across imbalances. 2 stars only had 466 reviews, but 5 stars had 5792, which means 5 star had more room for training. This could lead to misclassification between closely rated categories.

## Step 8: Reflection

Answer each question with reasoning and evidence from your results — not just one-sentence observations.

1. Which classifier + feature combo worked best for binary classification? Why?
2. Did stemming/lemmatization improve your results? Why or why not?
3. Did your embedding outperform BoW/TF-IDF? Why or why not?
4. Examine a few misclassified reviews — what made them hard to classify?
5. How does class imbalance affect your results?
6. What did you learn from your Step 7B choice (five-class or visualization)?
7. What one improvement would you add to this pipeline?

*Write your answers below in Markdown cells.*

1. The best binary classifier was Naive Bayes with Bow features because it had the highest overall (.903) and balanced accuracy (.806). It captured the negative class better than TF-IDF.

2. Lemmatization improved my results becaused it preserved word meaning while also reducing the size of the vocabulary.

3. My embedding did not outperform my BOW/TF-IDF. The Naive Bayes with BoW had .903 accuracy while the embedding had .895 and TF-IDF had .731. This was because BoW preserves sentiment words in numbers of appearance, which gets lost in logistic regression.

4. Ratings with mixed sentiment were hard to classify. An example would be when a review acknowledges a good thing and a bad thing about a product at the same time. It was easy for the model to pick up on positive words, but there were issues identifying and weighing the intensity behind certain word choice, as the model followed word patterns. Many reviews were predicted higher than they were due to this.

5. 5 star reviews made up 64% of all reviews. This plethora makes it easier for models to train accuracy with the biggest portion, while smaller classes are neglected. Class imbalance made the TF-IDF model look really good at first glance, but further inspection on the low balanced accuracy (.598) shows the negative effects of the class imbalance.

6. I learned from the 5 class experiment I chose in 7B that binary sentiment is not nearly as hard. In fine grained sentiment classification, there had to be separatoins in very small differences between things like 3 star ratings and 4 star ratings. This dropped accuracy from .90 in binary to .72 in 5 class.

7. An improvement I would add would be class weighting. The uneven distribution of reviews caused high favorability to the 5 star ratings, which effected recall for 1 and 2 star ratings since their proportion was so much smaller.  

---

## Before You Submit

- [ ] **Restart and Run All:** Kernel → Restart runtime, then Runtime → Run all. Every cell must run without errors.
- [ ] **All Markdown explanations are written** (Steps 3, 4A, 4B, 5, 6)
- [ ] **All reflection questions answered** with reasoning, not just observations
- [ ] **Confusion matrix + classification report** shown for every model
- [ ] **Balanced accuracy** reported alongside regular accuracy
- [ ] **Step 7B:** One option completed, the other deleted
- [ ] **Report:** 1–2 page PDF or Word document submitted separately

**Submit on Canvas by Sunday, March 29:** notebook (.ipynb) + report (PDF/Word)